# Qwen3.5 Image-to-SMT via Grammar-Constrained Decoding

> Originally, adapted from Qwen3_5_(0_8B)_Vision.ipynb

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import json
import random
from pathlib import Path

from IPython.display import Markdown, display
from PIL import Image

from staged_qwen3_5_scivqa.config import (
    BASE_DIR,
    COMPETITION_DATA_DIR,
    CVC5_PATH,
    MODEL_ID,
    PREAMBLE,
    EXAMPLES_PASS1A,
    EXAMPLES_PASS1B,
    EXAMPLES_PASS2,
    SMT_MAX_NEW_TOKENS,
    SMT_TEMPERATURE,
    SMT_TOP_P,
    SMT_TOP_K,
    SMT_MIN_P,
    SMT_REPETITION_PENALTY,
)
from staged_qwen3_5_scivqa.models.smt_runner import load_smt_model
from staged_qwen3_5_scivqa.smt.pipeline import (
    clean_duplicate_declarations,
    deduplicate_anchors,
    generate_declarations,
    parse_table_deterministically,
    reflect,
)

In [ ]:
CATEGORY = "train"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

In [ ]:
model = load_smt_model(MODEL_ID, max_new_tokens=SMT_MAX_NEW_TOKENS)

### Interactive sample selection

Pick a random sample of a given answer type and run the SMT reflection pipeline.

In [ ]:
ANSWER_TYPE = "Yes/No"

filename = None
q_index = None

image_index, count = random.randint(0, 100), 0
for file_path in CASE_DIR.rglob("*.json"):
    if (
        "content.json" in file_path.name
        or "images" not in str(file_path)
        or ".vscode" in str(file_path)
    ):
        continue

    with open(file_path) as f:
        data = json.load(f)

    sub_key = list(data["bbox"].keys())[0]

    if "vqa" not in data:
        continue

    if sub_key not in data["vqa"]:
        continue

    for i, q_obj in enumerate(data["vqa"][sub_key]):
        if q_obj.get("answer_type", "") == ANSWER_TYPE:
            filename = file_path
            q_index = i
            break

    if filename is not None:
        count += 1

    if count == image_index:
        break

In [ ]:
with filename.open("r") as f:
    data = json.load(f)

img = Image.open(filename.with_suffix(".jpg"))
box = data["bbox"][sub_key]

crop = img.crop((box["x"], box["y"], box["x"] + box["width"], box["y"] + box["height"]))
print(f"Subplot: {sub_key}")
for key, value in data["vqa"][sub_key][q_index].items():
    print(f"{key.replace('_', ' ').title()}: {value}")
display(crop)

In [ ]:
summary = data.get("summarization", {}).get(sub_key, "N/A")
table = data.get("data_extraction", {}).get(sub_key, "N/A")

md = f"""
# Summary
{summary}

# Table
{table}
"""

display(Markdown(md))

In [ ]:
reflect(
    model,
    data["vqa"][sub_key][q_index],
    crop,
    summary,
    table,
    max_retries=3,
    verbose=True,
    do_sample=True,
    max_new_tokens=SMT_MAX_NEW_TOKENS,
    temperature=SMT_TEMPERATURE,
    min_p=SMT_MIN_P,
    top_p=SMT_TOP_P,
    top_k=SMT_TOP_K,
    repetition_penalty=SMT_REPETITION_PENALTY,
)